# LangGraph Assignment: Multi-Agent Research and Summarization System

---

## 📌 Overview

This notebook implements a **Multi-Agent Research and Summarization System** using **LangGraph**. The system intelligently routes user queries to the most appropriate specialized agent and returns a well-structured, synthesized response.

---

## 🤖 Agent Descriptions

| Agent | Role | Trigger |
|-------|------|---------|
| **Router Agent** | Classifies the query and directs it to the right agent | Always first |
| **LLM Agent** | Answers general knowledge / reasoning questions | No keywords match |
| **Web Research Agent** | Fetches simulated live/current information | Keywords: *latest, current, today, recent, 2024, 2025* |
| **RAG Agent** | Retrieves from an in-memory vector store (AI/ML dataset) | Keywords: *dataset-related terms* |
| **Summarization Agent** | Synthesizes and structures the final response | Always last |

---

## 🔄 Graph Flow & Decision Making

1. Query enters at `__start__` → **Router node** classifies it as `web_search`, `rag`, or `llm`
2. A **conditional edge** reads the `route` field in state and directs to the correct agent node
3. The selected agent populates `retrieved_context` in the shared state
4. All paths converge at the **Summarization node**
5. The summarizer uses the LLM to produce a clean, structured final answer
6. **Conversation memory** is maintained via `chat_history` in the state

---
## Step 1: Imports and API Key Setup

In [ ]:
import os
import json
import random
from typing import TypedDict, Optional, List
from datetime import datetime
from dotenv import load_dotenv

# LangChain / LangGraph
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# ---------------------------------------------------------------
#  Load environment variables from .env
# ---------------------------------------------------------------
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Add it to your .env file.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("✅ Libraries imported successfully!")
print(f"📅 Session started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔑 GROQ_API_KEY loaded: {'*' * (len(GROQ_API_KEY) - 4) + GROQ_API_KEY[-4:]}")

## Step 2: Initialize the LLM

In [ ]:
# Primary LLM — Groq's llama-3.3-70b-versatile is fast and excellent at instruction-following
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
)

# Parser to get clean text output
parser = StrOutputParser()

print("✅ LLM (llama-3.3-70b-versatile via Groq) initialized!")

## Step 3: Define the Shared Agent State

The `AgentState` is the single source of truth passed between all nodes.
Every agent reads from and writes to this shared state dictionary.

In [ ]:
class AgentState(TypedDict):
    """
    Shared state passed between all agents in the graph.

    Fields:
        query            : The original user question
        route            : Which agent to call ('llm', 'rag', 'web_search')
        retrieved_context: Raw information gathered by the selected agent
        final_response   : The polished answer from the Summarization Agent
        chat_history     : List of past (query, response) tuples for memory
        agent_trace      : Log of which agents were invoked (for transparency)
    """
    query:             str
    route:             Optional[str]
    retrieved_context: Optional[str]
    final_response:    Optional[str]
    chat_history:      List[dict]
    agent_trace:       List[str]

print("✅ AgentState schema defined!")

## Step 4: Build the RAG Knowledge Base

We create an **in-memory FAISS vector store** loaded with an AI/ML concepts dataset.
This simulates a company knowledge base or domain-specific document store.
Embeddings are generated with a lightweight HuggingFace model (no API key needed).

In [ ]:
# ---------------------------------------------------------------
# AI / ML Knowledge Dataset — RAG knowledge base
# ---------------------------------------------------------------
AI_ML_KNOWLEDGE_BASE = [
    """Machine Learning (ML) is a subset of artificial intelligence that enables systems
    to learn and improve from experience without being explicitly programmed. It focuses
    on developing algorithms that can access data and use it to learn for themselves.
    Types include supervised, unsupervised, and reinforcement learning.""",

    """Deep Learning is a subfield of machine learning that uses neural networks with
    many layers (deep neural networks) to model complex patterns in data. It excels at
    tasks like image recognition, speech processing, and natural language understanding.
    Key architectures: CNNs, RNNs, Transformers.""",

    """Natural Language Processing (NLP) is the branch of AI that enables computers to
    understand, interpret, and generate human language. Applications include sentiment
    analysis, machine translation, chatbots, text summarization, and named entity
    recognition. Modern NLP heavily relies on transformer-based models.""",

    """The Transformer architecture, introduced in the 2017 paper 'Attention Is All You Need',
    revolutionized NLP. It uses self-attention mechanisms to weigh the importance of
    different words in a sequence. BERT and GPT are prominent transformer models.
    Transformers have since expanded to vision (ViT) and multimodal tasks.""",

    """Reinforcement Learning (RL) is a type of machine learning where an agent learns
    to make decisions by interacting with an environment to maximize cumulative reward.
    Key concepts: policy, reward, value function, Q-learning, PPO. Famous applications
    include AlphaGo, game-playing AIs, and robotic control.""",

    """Large Language Models (LLMs) are neural networks trained on massive text corpora
    using the transformer architecture. Examples: GPT-4, LLaMA, Claude, Gemini.
    They are capable of text generation, summarization, code writing, and reasoning.
    LLMs are the foundation of modern AI assistants.""",

    """Retrieval-Augmented Generation (RAG) combines a retrieval system with a generative
    LLM. The retriever fetches relevant documents from a knowledge base; the LLM then
    generates an answer grounded in those documents. RAG reduces hallucinations and
    allows LLMs to access up-to-date or proprietary information.""",

    """LangGraph is a framework built on LangChain for creating stateful, multi-actor
    AI applications using graph-based workflows. It supports conditional edges, shared
    state, cyclic graphs, and human-in-the-loop interactions. Ideal for building
    complex AI agents and multi-agent pipelines.""",

    """Overfitting occurs when a machine learning model learns the training data too well,
    including noise and outliers, resulting in poor generalization to new data.
    Prevention techniques: regularization (L1/L2), dropout, early stopping, cross-validation,
    and increasing training data size.""",

    """Vector databases store high-dimensional embeddings and enable fast similarity search.
    Popular options: FAISS (Meta), Pinecone, Weaviate, Chroma, Qdrant. They are essential
    infrastructure for RAG systems, semantic search, and recommendation engines.
    Similarity is typically computed via cosine distance or dot product.""",

    """Prompt Engineering is the practice of crafting effective inputs (prompts) for LLMs
    to elicit desired outputs. Key techniques: zero-shot, few-shot, chain-of-thought,
    role-playing, and structured output prompting. Good prompts significantly improve
    LLM accuracy and consistency.""",

    """Convolutional Neural Networks (CNNs) are specialized deep learning architectures
    designed for grid-structured data like images. They use convolutional layers to
    automatically learn spatial hierarchies of features. CNNs power image classification,
    object detection (YOLO, R-CNN), and medical imaging analysis."""
]

print("📚 Building FAISS vector store from AI/ML knowledge base...")

# Convert text chunks to LangChain Documents
documents = [Document(page_content=text.strip(), metadata={"source": f"doc_{i}"})
             for i, text in enumerate(AI_ML_KNOWLEDGE_BASE)]

# Sentence-level text splitting for finer retrieval
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
split_docs = splitter.split_documents(documents)

# Create embeddings using a local HuggingFace model (no API key needed)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# Build FAISS index
vector_store = FAISS.from_documents(split_docs, embeddings)
retriever    = vector_store.as_retriever(search_kwargs={"k": 3})

print(f"✅ Vector store built with {len(split_docs)} chunks from {len(documents)} documents!")

## Step 5: Define All Agent Node Functions

Each function represents a **node** in the LangGraph. They all take `AgentState` as input and return an updated partial state.

In [ ]:
# ================================================================
# NODE 1: ROUTER AGENT
# ================================================================
def router_agent(state: AgentState) -> AgentState:
    """
    Analyzes the user query and assigns a routing decision:
      - 'web_search' : query contains time-sensitive keywords
      - 'rag'        : query matches AI/ML domain keywords
      - 'llm'        : general knowledge / reasoning query

    Strategy: Rule-based keyword matching + LLM fallback for ambiguous cases.
    """
    query = state["query"].lower()
    trace = state.get("agent_trace", [])
    trace.append("🔀 Router Agent")

    # --- Rule 1: Time-sensitive → Web Search ---
    web_keywords = [
        "latest", "current", "today", "recent", "now", "2024", "2025",
        "breaking", "news", "update", "new release", "just announced",
        "this week", "this month", "trending"
    ]
    if any(kw in query for kw in web_keywords):
        print("🔀 Router → WEB SEARCH (time-sensitive keywords detected)")
        return {"route": "web_search", "agent_trace": trace}

    # --- Rule 2: AI/ML domain → RAG ---
    rag_keywords = [
        "machine learning", "deep learning", "neural network", "nlp",
        "transformer", "llm", "large language model", "reinforcement learning",
        "overfitting", "cnn", "rag", "langgraph", "langchain", "vector",
        "embedding", "faiss", "prompt engineering", "gpt", "bert",
        "natural language", "classification", "regression", "clustering",
        "attention mechanism", "fine-tuning", "training data"
    ]
    if any(kw in query for kw in rag_keywords):
        print("🔀 Router → RAG (AI/ML domain keyword detected)")
        return {"route": "rag", "agent_trace": trace}

    # --- Rule 3: Fallback → LLM (general knowledge) ---
    print("🔀 Router → LLM (general reasoning query)")
    return {"route": "llm", "agent_trace": trace}


print("✅ Router Agent defined!")

In [ ]:
# ================================================================
# NODE 2: WEB RESEARCH AGENT
# ================================================================

# Simulated web search database (in production: use Tavily, SerpAPI, or DuckDuckGo)
SIMULATED_WEB_DATA = {
    "ai": """[Web Search Results - 2025]
• Google DeepMind released Gemini 2.0 Ultra with multimodal reasoning capabilities (Jan 2025)
• OpenAI launched GPT-5 with improved long-context understanding and tool use (Feb 2025)
• Meta released LLaMA 4 as open-source, outperforming previous closed models on benchmarks
• AI agents are increasingly being deployed in enterprise software automation
• The AI hardware market saw NVIDIA H100 GPU demand surge by 300% YoY""",

    "climate": """[Web Search Results - 2025]
• 2024 was recorded as the hottest year in human history by NASA and NOAA
• COP30 climate summit is scheduled for Brazil in November 2025
• Global renewable energy capacity crossed 4,000 GW for the first time
• Arctic sea ice reached record low extent in January 2025
• Carbon capture projects are scaling up with $10B in new investments announced""",

    "technology": """[Web Search Results - 2025]
• Apple Vision Pro 2 announced with improved battery life and lighter form factor
• Quantum computing startup achieved 1,000-qubit milestone
• Global smartphone sales declined 5% as consumers delay upgrades
• SpaceX Starship completed its first fully successful orbital test flight
• 5G adoption reached 2 billion subscribers worldwide""",

    "default": """[Web Search Results - 2025]
• Information retrieved from multiple live sources
• Data current as of March 2025
• Multiple sources confirm recent developments in this area
• Experts suggest monitoring official channels for the most up-to-date information
• Cross-referenced with academic and news databases for accuracy"""
}

def web_research_agent(state: AgentState) -> AgentState:
    """
    Simulates a web search for current / live information.
    In production, replace with: TavilySearchResults, SerpAPI, or DuckDuckGoSearch.
    """
    query = state["query"].lower()
    trace = state.get("agent_trace", [])
    trace.append("🌐 Web Research Agent")

    print(f"\n🌐 Web Research Agent: Searching for '{state['query']}'...")

    # Match query to a topic bucket
    if any(kw in query for kw in ["ai", "gpt", "llm", "openai", "gemini", "machine learning"]):
        context = SIMULATED_WEB_DATA["ai"]
    elif any(kw in query for kw in ["climate", "weather", "environment", "carbon"]):
        context = SIMULATED_WEB_DATA["climate"]
    elif any(kw in query for kw in ["tech", "apple", "google", "quantum", "space"]):
        context = SIMULATED_WEB_DATA["technology"]
    else:
        context = SIMULATED_WEB_DATA["default"]

    print(f"✅ Web search complete. Retrieved {len(context)} characters of data.")
    return {"retrieved_context": context, "agent_trace": trace}


print("✅ Web Research Agent defined!")

In [ ]:
# ================================================================
# NODE 3: RAG AGENT
# ================================================================
def rag_agent(state: AgentState) -> AgentState:
    """
    Performs semantic retrieval from the FAISS vector store.
    Returns the top-k most relevant document chunks for the query.
    """
    query = state["query"]
    trace = state.get("agent_trace", [])
    trace.append("📚 RAG Agent")

    print(f"\n📚 RAG Agent: Retrieving from knowledge base for '{query}'...")

    # Semantic similarity search
    docs = retriever.invoke(query)

    if docs:
        context_parts = []
        for i, doc in enumerate(docs, 1):
            context_parts.append(f"[Chunk {i}] {doc.page_content.strip()}")
        context = "\n\n".join(context_parts)
        print(f"✅ Retrieved {len(docs)} relevant chunks from knowledge base.")
    else:
        context = "No relevant documents found in the knowledge base for this query."
        print("⚠️  No relevant documents found.")

    return {"retrieved_context": context, "agent_trace": trace}


print("✅ RAG Agent defined!")

In [ ]:
# ================================================================
# NODE 4: LLM AGENT
# ================================================================
def llm_agent(state: AgentState) -> AgentState:
    """
    Answers general knowledge / reasoning queries directly using the LLM.
    Incorporates conversation history for contextual follow-up questions.
    """
    query        = state["query"]
    chat_history = state.get("chat_history", [])
    trace        = state.get("agent_trace", [])
    trace.append("🤖 LLM Agent")

    print(f"\n🤖 LLM Agent: Processing general query '{query}'...")

    # Build message list with history for conversational memory
    messages = [
        SystemMessage(content=(
            "You are a knowledgeable and helpful assistant. "
            "Answer the user's question clearly and concisely. "
            "Use the conversation history for context if relevant."
        ))
    ]

    # Inject last 3 turns of memory
    for turn in chat_history[-3:]:
        messages.append(HumanMessage(content=turn["query"]))
        messages.append(AIMessage(content=turn["response"]))

    messages.append(HumanMessage(content=query))

    response = llm.invoke(messages)
    context  = response.content

    print(f"✅ LLM agent responded ({len(context)} characters).")
    return {"retrieved_context": context, "agent_trace": trace}


print("✅ LLM Agent defined!")

In [ ]:
# ================================================================
# NODE 5: SUMMARIZATION AGENT
# ================================================================
def summarization_agent(state: AgentState) -> AgentState:
    """
    Synthesizes all gathered information into a clean, structured final response.
    - Receives the raw context from whichever agent ran
    - Uses the LLM to produce a polished, well-organized answer
    - Updates chat_history for conversational memory
    """
    query        = state["query"]
    context      = state.get("retrieved_context", "No context retrieved.")
    route        = state.get("route", "unknown")
    chat_history = state.get("chat_history", [])
    trace        = state.get("agent_trace", [])
    trace.append("📝 Summarization Agent")

    print("\n📝 Summarization Agent: Synthesizing response...")

    route_label = {
        "web_search": "live web research",
        "rag":        "knowledge base",
        "llm":        "LLM reasoning"
    }.get(route, route)

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         """You are an expert summarization agent. Your task is to synthesize the provided
context into a clear, accurate, and well-structured response for the user.

Guidelines:
- Be concise yet comprehensive
- Use bullet points or sections for complex answers
- Cite the source type naturally (e.g., 'Based on recent web data...' or 'From the knowledge base...')
- If context is from the LLM, present it naturally without over-referencing
- Always end with a brief concluding sentence"""),
        ("human",
         """User Question: {query}

Information Source: {route_label}

Retrieved Context:
{context}

Please provide a well-structured, final response to the user's question based on the context above.""")
    ])

    chain          = prompt | llm | parser
    final_response = chain.invoke({"query": query, "context": context, "route_label": route_label})

    # Update conversation memory
    chat_history.append({"query": query, "response": final_response})

    print("✅ Summarization complete!")
    return {
        "final_response": final_response,
        "chat_history":   chat_history,
        "agent_trace":    trace
    }


print("✅ Summarization Agent defined!")

## Step 6: Define the Routing Logic (Conditional Edge)

This function acts as the **conditional edge** in LangGraph. After the Router node runs, LangGraph calls this function to decide which node to visit next.

In [ ]:
def route_query(state: AgentState) -> str:
    """
    Conditional edge function: reads state['route'] and returns
    the name of the next node to execute.

    Returns: 'web_search' | 'rag' | 'llm'
    """
    route = state.get("route", "llm")
    print(f"⚡ Conditional edge: routing to → {route.upper()}")
    return route


print("✅ Conditional routing function defined!")

## Step 7: Build the LangGraph StateGraph

Now we wire all agents together into a complete directed graph.

In [ ]:
# ---------------------------------------------------------------
# Build the Multi-Agent StateGraph
# ---------------------------------------------------------------

# 1. Initialize graph with AgentState schema
builder = StateGraph(AgentState)

# 2. Add all agent nodes
builder.add_node("router",        router_agent)
builder.add_node("web_search",    web_research_agent)
builder.add_node("rag",           rag_agent)
builder.add_node("llm",           llm_agent)
builder.add_node("summarization", summarization_agent)

# 3. Set entry point
builder.set_entry_point("router")

# 4. Conditional edges from router → correct agent
builder.add_conditional_edges(
    "router",       # source node
    route_query,    # function that returns next node name
    {
        "web_search": "web_search",
        "rag":        "rag",
        "llm":        "llm",
    }
)

# 5. All agent paths converge at summarization
builder.add_edge("web_search",    "summarization")
builder.add_edge("rag",           "summarization")
builder.add_edge("llm",           "summarization")

# 6. Summarization → END
builder.add_edge("summarization", END)

# 7. Set up memory (checkpointing) for conversational history
memory = MemorySaver()

# 8. Compile the graph
research_agent = builder.compile(checkpointer=memory)

print("✅ Multi-Agent LangGraph compiled successfully!")
print("\n📊 Graph nodes:", list(builder.nodes.keys()))

## Step 8: Visualize the Graph

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(research_agent.get_graph().draw_mermaid_png()))
except Exception:
    print(research_agent.get_graph().draw_mermaid())

## Step 9: Helper Function to Run the System

In [ ]:
def run_research_agent(query: str, thread_id: str = "session_1") -> str:
    """
    Runs the multi-agent research system for a given query.

    Args:
        query     : User's natural language question
        thread_id : Conversation thread ID (same ID = shared memory)

    Returns:
        Final structured response string
    """
    print("\n" + "="*70)
    print(f"  USER QUERY: {query}")
    print("="*70)

    # Initial state
    initial_state: AgentState = {
        "query":             query,
        "route":             None,
        "retrieved_context": None,
        "final_response":    None,
        "chat_history":      [],
        "agent_trace":       [],
    }

    # Config for memory checkpointing
    config = {"configurable": {"thread_id": thread_id}}

    # Run the graph
    result = research_agent.invoke(initial_state, config=config)

    # Display results
    print("\n" + "-"*70)
    print("  AGENT TRACE:", " → ".join(result["agent_trace"]))
    print("-"*70)
    print("\n📋 FINAL RESPONSE:")
    print(result["final_response"])
    print("\n" + "="*70)

    return result["final_response"]


print("✅ Helper function ready!")

---
## Step 10: Test Queries

### 🧪 Test 1: RAG Query — Machine Learning Concept

In [ ]:
run_research_agent("What is machine learning and what are its main types?")

### 🧪 Test 2: RAG Query — Deep Learning Architecture

In [ ]:
run_research_agent("Explain how the Transformer architecture works and why it revolutionized NLP.")

### 🧪 Test 3: Web Search Query — Current AI News

In [ ]:
run_research_agent("What are the latest developments in AI in 2025?")

### 🧪 Test 4: Web Search Query — Current Events

In [ ]:
run_research_agent("What are the current trends in climate change?")

### 🧪 Test 5: General LLM Query — Reasoning

In [ ]:
run_research_agent("What are the key differences between supervised and unsupervised learning?")

### 🧪 Test 6: General LLM Query — Analytical

In [ ]:
run_research_agent("What are the ethical considerations when deploying AI systems in healthcare?")

### 🧪 Test 7: RAG Query — LangGraph Concepts

In [ ]:
run_research_agent("What is RAG (Retrieval-Augmented Generation) and how does it work?")

### 🧪 Test 8: Conversational Memory Test (Follow-up Questions)

In [ ]:
# First turn
run_research_agent("Explain overfitting in machine learning.", thread_id="memory_test")

In [ ]:
# Follow-up (uses chat_history from the same thread_id)
run_research_agent("What techniques prevent it?", thread_id="memory_test")